In [1]:
import numpy as np
import math

def surrogate_value(k, J, a):
    # grid point x = k / 10^J
    # k = 0, 1, ..., 10^J

    if k == 10**J:
        digits = [9] * J
        z = 1
    else:
        digits = [int(ch) for ch in f"{k:0{J}d}"]
        z = 0

    value = 0.0
    sum_all_P = 0.0

    for j in range(1, J + 1):
        for i in range(1, 10):
            P = (i * 10**(-j))**a - ((i - 1) * 10**(-j))**a
            sum_all_P += P

            if i <= digits[j - 1]:
                value += P

    Q = 1.0 - sum_all_P
    value += Q * z

    return value

def error_table(exponents, Js):
    rows = []

    for a in exponents:
        for J in Js:
            xs = np.arange(0, 10**J + 1) / 10**J
            hats = np.array([surrogate_value(k, J, a) for k in range(10**J + 1)])
            true = xs**a
            err = hats - true

            max_idx = np.argmax(np.abs(err))

            rows.append({
                "a": a,
                "J": J,
                "grid_size": len(xs),
                "max_abs_error": np.max(np.abs(err)),
                "rmse": math.sqrt(np.mean(err**2)),
                "max_error_point": xs[max_idx],
                "surrogate_at_max": hats[max_idx],
                "true_at_max": true[max_idx],
            })

    return rows

rows = error_table(exponents=[0.6, 1.5], Js=[1, 2, 3])

for r in rows:
    print(r)

{'a': 0.6, 'J': 1, 'grid_size': 11, 'max_abs_error': 1.1102230246251565e-16, 'rmse': 3.7425610525290256e-17, 'max_error_point': 0.6, 'surrogate_at_max': 0.7360219228178334, 'true_at_max': 0.7360219228178333}
{'a': 0.6, 'J': 2, 'grid_size': 101, 'max_abs_error': 0.180553375376858, 'rmse': 0.10947070387325103, 'max_error_point': 0.99, 'surrogate_at_max': 1.1745413190385563, 'true_at_max': 0.9939879436616983}
{'a': 0.6, 'J': 3, 'grid_size': 1001, 'max_abs_error': 0.23437195366963415, 'rmse': 0.13947465241652124, 'max_error_point': 0.999, 'surrogate_at_max': 1.2337718336136005, 'true_at_max': 0.9993998799439664}
{'a': 1.5, 'J': 1, 'grid_size': 11, 'max_abs_error': 1.1102230246251565e-16, 'rmse': 5.021172554038838e-17, 'max_error_point': 0.6, 'surrogate_at_max': 0.4647580015448901, 'true_at_max': 0.46475800154489}
{'a': 1.5, 'J': 2, 'grid_size': 101, 'max_abs_error': 0.10422259449009119, 'rmse': 0.042623506543928644, 'max_error_point': 0.99, 'surrogate_at_max': 0.8808149682454626, 'true_at_

In [ ]:
import numpy as np
import math

def compute_coefficients(J, a):
    """
    Computes P_{j,i} and Q for the surrogate

        x^a ≈ sum_{j=1}^J sum_{i=1}^9 P_{j,i} z_{j,i} + Q z

    where

        P_{j,i} = (i * 10^{-j})^a - ((i-1) * 10^{-j})^a
    """

    coeffs = []
    sum_all_P = 0.0

    for j in range(1, J + 1):
        for i in range(1, 10):
            P = (i * 10**(-j))**a - ((i - 1) * 10**(-j))**a
            coeffs.append({
                "j": j,
                "i": i,
                "P": P
            })
            sum_all_P += P

    Q = 1.0 - sum_all_P

    return coeffs, Q


def print_coefficients(J, a, decimals=6):
    coeffs, Q = compute_coefficients(J, a)

    print(f"\nCoefficients for a = {a}, J = {J}")
    print("-" * 45)

    for row in coeffs:
        j = row["j"]
        i = row["i"]
        P = row["P"]

        if J == 1:
            # For J = 1, this corresponds to P_1, ..., P_9
            print(f"P_{i} = {P:.{decimals}f}")
        else:
            # For J > 1, use digit-indexed notation
            print(f"P_{{{j},{i}}} = {P:.{decimals}f}")

    print(f"Q = {Q:.{decimals}f}")


def surrogate_value(k, J, a):
    # grid point x = k / 10^J
    # k = 0, 1, ..., 10^J

    if k == 10**J:
        digits = [9] * J
        z = 1
    else:
        digits = [int(ch) for ch in f"{k:0{J}d}"]
        z = 0

    value = 0.0
    coeffs, Q = compute_coefficients(J, a)

    for row in coeffs:
        j = row["j"]
        i = row["i"]
        P = row["P"]

        if i <= digits[j - 1]:
            value += P

    value += Q * z

    return value


def error_table(exponents, Js):
    rows = []

    for a in exponents:
        for J in Js:
            xs = np.arange(0, 10**J + 1) / 10**J

            hats = np.array([
                surrogate_value(k, J, a)
                for k in range(10**J + 1)
            ])

            true = xs**a
            err = hats - true

            max_idx = np.argmax(np.abs(err))

            rows.append({
                "a": a,
                "J": J,
                "grid_size": len(xs),
                "max_abs_error": np.max(np.abs(err)),
                "rmse": math.sqrt(np.mean(err**2)),
                "max_error_point": xs[max_idx],
                "surrogate_at_max": hats[max_idx],
                "true_at_max": true[max_idx],
            })

    return rows



# Print coefficients

for a in [0.6, 1.5]:
    for J in [1, 2, 3]:
        print_coefficients(J, a, decimals=6)


# Print error table

rows = error_table(exponents=[0.6, 1.5], Js=[1, 2, 3])

print("\nError table")
print("-" * 95)

for r in rows:
    print(
        f"a = {r['a']}, "
        f"J = {r['J']}, "
        f"grid size = {r['grid_size']}, "
        f"max abs error = {r['max_abs_error']:.6e}, "
        f"RMSE = {r['rmse']:.6e}, "
        f"max-error point = {r['max_error_point']:.6f}, "
        f"surrogate = {r['surrogate_at_max']:.6f}, "
        f"true = {r['true_at_max']:.6f}"
    )


Coefficients for a = 0.6, J = 1
---------------------------------------------
P_1 = 0.251189
P_2 = 0.129542
P_3 = 0.104863
P_4 = 0.091487
P_5 = 0.082674
P_6 = 0.076268
P_7 = 0.071322
P_8 = 0.067345
P_9 = 0.064051
Q = 0.061260

Coefficients for a = 0.6, J = 2
---------------------------------------------
P_{1,1} = 0.251189
P_{1,2} = 0.129542
P_{1,3} = 0.104863
P_{1,4} = 0.091487
P_{1,5} = 0.082674
P_{1,6} = 0.076268
P_{1,7} = 0.071322
P_{1,8} = 0.067345
P_{1,9} = 0.064051
P_{2,1} = 0.063096
P_{2,2} = 0.032540
P_{2,3} = 0.026340
P_{2,4} = 0.022980
P_{2,5} = 0.020767
P_{2,6} = 0.019158
P_{2,7} = 0.017915
P_{2,8} = 0.016916
P_{2,9} = 0.016089
Q = -0.174541

Coefficients for a = 0.6, J = 3
---------------------------------------------
P_{1,1} = 0.251189
P_{1,2} = 0.129542
P_{1,3} = 0.104863
P_{1,4} = 0.091487
P_{1,5} = 0.082674
P_{1,6} = 0.076268
P_{1,7} = 0.071322
P_{1,8} = 0.067345
P_{1,9} = 0.064051
P_{2,1} = 0.063096
P_{2,2} = 0.032540
P_{2,3} = 0.026340
P_{2,4} = 0.022980
P_{2,5} = 0.